# Preprocessing EU-LFS data



### Notebook purpose
1. Merge yearly country files and prepare for simulation

#### 1. Set-up

In [1]:
import sys
import os
from pathlib import Path

# Set the working directory to project root
project_root = Path().resolve().parents[0]

# Add 'src' directory
sys.path.append(str(project_root / "src"))
sys.path.append(str(project_root / "data"))
print(sys.path)

['/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys', '/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev', '/Applications/PyCharm.app/Contents/plugins/python/helpers-pro/jupyter_debug', '/opt/homebrew/Cellar/python@3.11/3.11.11/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.11/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.11/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/.venv/lib/python3.11/site-packages', '/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/notebooks/src', '/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/notebooks/data']


In [2]:
# import packages and modules

import numpy as np
import pandas as pd

from data.framework import Esco, Classifications
from data.lfs import EuLfs
from src import utils

# Load central paths object
useful_paths = utils.UsefulPaths(fn_config_path="paths_config.yml")
lfs_config = utils.load_config(os.path.join(useful_paths.config_dir, "eu_lfs_config.yml"))

# Parameter
year = 2023

#### 2. Load metadata

Geodata (loaded through lfs.py)

NACE labels

In [3]:
classifications = Classifications()
covariates_by_nace = classifications.nace_1d

Occupation classification (Zaussinger et al., 2025)

In [4]:
# Occupation classification (Zaussinger et al. 2025)
esco = Esco()
ndigits = 3

gbn_shares_no_wt = esco.read_gbn_classification(agg_to_isco_at_digit=ndigits)

# Rename to match LFS
gbn_shares_no_wt = gbn_shares_no_wt.rename(columns={
    "preferredLabel_isco": "ISCO08_3D_label",
    "isco_code":        "ISCO08_3D",
})

# Keep zfilled code and weights for simulation
gbn_shares_no_wt["code"] = (
    gbn_shares_no_wt["ISCO08_3D"]
      .astype(str)
      .str.zfill(ndigits)
)
gbn_shares_no_wt["NOBS"] = 1

# define the classification category per ISCO as the one with the highest fraction
share_cols = [
    "share_low-carbon",
    "share_neutral",
    "share_viable-to-decarbonize",
    "share_unviable-to-decarbonize",
    "share_high-carbon",
]

gbn_shares_no_wt["category"] = gbn_shares_no_wt[share_cols].idxmax(axis=1)

In [5]:
# save

gbn_shares_no_wt.to_csv(
    os.path.join(
        useful_paths.data_processed,
        "esco",
        "final_gbn_shares_by_isco3d_unweighted_new.csv".format(ndigits),
    )
)

In [6]:
# see which occs will be selected in which simulation job pool

df_shares = gbn_shares_no_wt

# ompute the global means (just like reskilling.py does per country)
mean_unv   = df_shares["share_unviable-to-decarbonize"].mean()
mean_viab  = df_shares["share_viable-to-decarbonize"].mean()
mean_neut  = df_shares["share_neutral"].mean()
mean_low   = df_shares["share_low-carbon"].mean()
# (high-carbon is a sum, no need for its mean)

# build each pool exactly as in the reskilling.py logic

# at-risk: unviable > its mean
pool_at_risk = df_shares.loc[
    df_shares["share_unviable-to-decarbonize"] > mean_unv,
    ["ISCO08_3D", "ISCO08_3D_label"]
].assign(pool="at-risk")

# high-carbon: viable OR unviable > their means
pool_high_carbon = df_shares.loc[
    (df_shares["share_viable-to-decarbonize"]   > mean_viab) |
    (df_shares["share_unviable-to-decarbonize"] > mean_unv),
    ["ISCO08_3D", "ISCO08_3D_label"]
].assign(pool="high-carbon")

# shortage: neutral OR low-carbon > their means
pool_shortage = df_shares.loc[
    (df_shares["share_neutral"]      > mean_neut) |
    (df_shares["share_low-carbon"]   > mean_low),
    ["ISCO08_3D", "ISCO08_3D_label"]
].assign(pool="shortage")

# stitch together and save
all_pools = pd.concat([pool_at_risk, pool_high_carbon, pool_shortage], ignore_index=True)
out_path = os.path.join(
    useful_paths.data_processed,
    "esco",
    "job_pools_isco3d_by_scenario.csv"
)
all_pools.to_csv(out_path, index=False)

print(f"Wrote {len(all_pools)} rows to {out_path}")
print(all_pools.groupby("pool").size())

Wrote 156 rows to /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/processed/esco/job_pools_isco3d_by_scenario.csv
pool
at-risk         18
high-carbon     24
shortage       114
dtype: int64


Eurostat SILC earnings data

In [7]:
# Define path and load
income_csv_path = os.path.join(useful_paths.data_raw, "metadata", "eurostat_income_deciles_2023.csv")
df_income_raw = pd.read_csv(income_csv_path)

# Filter only decile-based earnings (indic_il = "TC")
df_income_raw = df_income_raw[df_income_raw["indic_il"] == "TC"].copy()

# Extract decile number from 'quantile' column (e.g., D1 → 1.0)
df_income_raw["INCDECIL"] = df_income_raw["quantile"].str.extract(r"D(\d+)").astype(float)

# Extract country code
df_income_raw["COUNTRYW"] = df_income_raw["geo"]

# Extract and convert annual earnings value
df_income_raw["income_eur"] = pd.to_numeric(df_income_raw["OBS_VALUE"], errors="coerce")

# Keep only necessary columns
df_income = df_income_raw[["COUNTRYW", "INCDECIL", "income_eur"]].dropna()

In [8]:
# Impute 10th decile using D9-D9 multiplier based on German data from 2016 -> update!!

# Path to 2016 German decile earnings
de_2023_path = os.path.join(useful_paths.data_raw, "metadata", "eurostat_income_deciles_2023_DE.csv")
df_de_2023 = pd.read_csv(de_2023_path)

# Compute multiplier from 2016 data (columns are already numeric)
d9 = df_de_2023[df_de_2023["decile"] == 9]["annual_earnings"].values[0]
d10 = df_de_2023[df_de_2023["decile"] == 10]["annual_earnings"].values[0]
d9_to_d10_multiplier = d10 / d9
print(f"D9→D10 multiplier (Germany 2023): {d9_to_d10_multiplier:.2f}")

# Apply multiplier to current earnings data
df_d9 = df_income[df_income["INCDECIL"] == 9].copy()
df_d10_imputed = df_d9.copy()
df_d10_imputed["INCDECIL"] = 10
df_d10_imputed["income_eur"] = df_d10_imputed["income_eur"] * d9_to_d10_multiplier

# Combine and sort
df_income_full = pd.concat([df_income, df_d10_imputed], ignore_index=True)
df_income_full = df_income_full.sort_values(by=["COUNTRYW", "INCDECIL"]).reset_index(drop=True)

D9→D10 multiplier (Germany 2023): 1.37


#### 3. Create dataset for simulation

Load and combine individual LFS files

In [9]:
%%time

# Instantiate class
eulfs = EuLfs(
    useful_paths=useful_paths,
    config=lfs_config
)

# Preprocess data
eulfs.preprocess_files(
    years=[2023],
    countries=[
        "AT", "BE", "BG", "CH", "CY", "CZ", "DE", "DK", "EL", "EE", "ES", "FI", "FR",
        "HR", "HU", "IE", "IS", "IT", "LT", "LU", "LV", "MT", "NL", "NO", "PL", "PT",
        "RO", "SE", "SI", "SK"
    ],
    save_file=True  # saves automatically
)

# Load preprocessed data
df_lfs = eulfs.read_preprocessed_file(year=2023, ffmt="csv")

2023
AT


/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/lfs.py:368: UserWarning: AT-2023: dropping 425 obsolete NUTS2 codes
  warnings.warn(f"{country}-{year}: dropping {n_obsolete} obsolete "
/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/lfs.py:368: UserWarning: BE-2023: dropping 64 obsolete NUTS2 codes
  warnings.warn(f"{country}-{year}: dropping {n_obsolete} obsolete "


Kept rows after all filters: 87090
BE
Kept rows after all filters: 17468
BG


/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/lfs.py:368: UserWarning: BG-2023: dropping 130 obsolete NUTS2 codes
  warnings.warn(f"{country}-{year}: dropping {n_obsolete} obsolete "
/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/lfs.py:368: UserWarning: CH-2023: dropping 6370 obsolete NUTS2 codes
  warnings.warn(f"{country}-{year}: dropping {n_obsolete} obsolete "


Kept rows after all filters: 13425
CH
Kept rows after all filters: 36567
CY
Kept rows after all filters: 17622
CZ


...

Kept rows after all filters: 9110
RO


/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/lfs.py:368: UserWarning: RO-2023: dropping 1226 obsolete NUTS2 codes
  warnings.warn(f"{country}-{year}: dropping {n_obsolete} obsolete "


Kept rows after all filters: 83840
SE


/Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/lfs.py:368: UserWarning: SE-2023: dropping 543 obsolete NUTS2 codes
  warnings.warn(f"{country}-{year}: dropping {n_obsolete} obsolete "


Kept rows after all filters: 63373
SI
Kept rows after all filters: 24001
SK
Kept rows after all filters: 37185
CPU times: user 13.3 s, sys: 899 ms, total: 14.2 s
Wall time: 14.4 s


Impute missing INCDECIL data

In [10]:
# Ensure INCDECIL is float
df_lfs["INCDECIL"] = df_lfs["INCDECIL"].astype(float)

# Compute group-level median decile values
group_medians = (
    df_lfs.groupby(["COUNTRYW", "ISCO08_3D", "NACE2_1D"], observed=True)["INCDECIL"]
    .median()
    .reset_index()
    .rename(columns={"INCDECIL": "INCDECIL_imputed"})
)

# Merge medians into main data
df_lfs = pd.merge(
    df_lfs,
    group_medians,
    on=["COUNTRYW", "ISCO08_3D", "NACE2_1D"],
    how="left",
)

# Fill in final income decile variable
df_lfs["INCDECIL_final"] = df_lfs["INCDECIL"]
df_lfs.loc[df_lfs["INCDECIL_final"].isna(), "INCDECIL_final"] = df_lfs["INCDECIL_imputed"]

Merge metadata

In [11]:
# Merge occupation classification
df_lfs = pd.merge(df_lfs, gbn_shares_no_wt, how="left", left_on="ISCO08_3D", right_on="ISCO08_3D")

for col in df_lfs.columns: # compute weighted GBN shares by multiplying with COEFFY
    if col.startswith("share_"):
        df_lfs[f"COEFFY_{col}"] = df_lfs["COEFFY"] * df_lfs[col]

# Merge NACE labels
df_lfs = pd.merge(df_lfs, covariates_by_nace, how="left", on="NACE2_1D")

# Merge earnings data
df_income_full["INCDECIL"] = df_income_full["INCDECIL"].astype(float)
df_income_full["COUNTRYW"] = df_income_full["COUNTRYW"].astype(str).str.strip()
df_lfs["INCDECIL"] = df_lfs["INCDECIL"].astype(float)
df_lfs["COUNTRYW"] = df_lfs["COUNTRYW"].astype(str).str.strip()

df_lfs = pd.merge(
    df_lfs,
    df_income_full.rename(columns={"INCDECIL": "INCDECIL_final"}),  # align naming
    how="left",
    on=["COUNTRYW", "INCDECIL_final"]
)

In [12]:
df_lfs

,REFYEAR,HHTYPE,COEFFY,COUNTRY,REGION_2D,DEGURBA,SEX,AGE,WKSTAT,ILOSTAT,...,code,NOBS,category,COEFFY_share_low-carbon,COEFFY_share_neutral,COEFFY_share_viable-to-decarbonize,COEFFY_share_unviable-to-decarbonize,COEFFY_share_high-carbon,"NACE2_1D_label,,",income_eur
0,2023,1,15.4400,AT,10,2,1,57,2,1,...,243,1.0,share_neutral,0.643333,14.796667,0.0,0.0,0.0,NaN,NaN
1,2023,1,15.4400,AT,10,2,2,54,2,1,...,111,1.0,share_neutral,0.000000,15.440000,0.0,0.0,0.0,NaN,53527.0
2,2023,1,15.2200,AT,10,2,2,55,1,1,...,334,1.0,share_neutral,0.000000,15.220000,0.0,0.0,0.0,NaN,NaN
3,2023,1,15.2200,AT,10,2,1,57,1,1,...,541,1.0,share_neutral,0.000000,15.220000,0.0,0.0,0.0,NaN,NaN
4,2023,1,25.2600,AT,10,2,1,43,1,1,...,742,1.0,share_neutral,0.000000,25.260000,0.0,0.0,0.0,"Manufacturing,,",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171337,2023,1,88.5216,SK,04,2,2,<NA>,1,1,...,331,1.0,share_neutral,0.000000,88.521600,0.0,0.0,0.0,"Education,,",7665.0
1171338,2023,1,88.5216,SK,04,2,1,<NA>,1,1,...,522,1.0,share_neutral,0.000000,88.521600,0.0,0.0,0.0,NaN,8432.0
1171339,2023,1,78.6013,SK,04,2,2,<NA>,1,1,...,512,1.0,share_neutral,0.000000,78.601300,0.0,0.0,0.0,"Accommodation and Food Service Activities,,",7665.0
1171340,2023,1,78.6013,SK,04,2,1,<NA>,1,1,...,833,1.0,share_neutral,7.145573,71.455727,0.0,0.0,0.0,"Transportation and Storage,,",9214.0


In [13]:
# format cleaning

df_lfs = df_lfs.rename(columns={"income_eur": "annual_earnings"})
df_lfs = df_lfs.loc[:, ~df_lfs.columns.str.endswith("_y")]
df_lfs.columns = df_lfs.columns.str.replace(r"_x$", "", regex=True)
df_lfs["NACE2_1D_label"] = df_lfs["NACE2_1D_label,,"].str.strip(" ,")
df_lfs = df_lfs.drop(columns=["NACE2_1D_label,,"], errors="ignore")

In [13]:
year = 2023

# Save final output file for reskilling model
output_fname = f"eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed.csv"
final_output_path = os.path.join(useful_paths.path_eulfs_interim, output_fname)
df_lfs.to_csv(final_output_path, index=False)

pkl_fname = f"eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed.pkl"
final_pkl_path = os.path.join(useful_paths.path_eulfs_interim, pkl_fname)
df_lfs.to_pickle(final_pkl_path)

Aggregate shares and save extra for simulation

In [14]:
# Aggregation dictionary
agg_dict = {
    "COEFFY": "sum",
    "NOBS": "sum",
    "COEFFY_share_low-carbon":            "sum",
    "COEFFY_share_neutral":               "sum",
    "COEFFY_share_viable-to-decarbonize": "sum",
    "COEFFY_share_unviable-to-decarbonize": "sum",
    "COEFFY_share_high-carbon":           "sum",
}

# Aggregate by industry (NACE 1D label)
df_nace = df_lfs.groupby("NACE2_1D_label").agg(agg_dict).reset_index()
df_nace.to_csv(
    os.path.join(
        useful_paths.path_eulfs_processed,
        f"eulfs_{year}_by_nace1d_final_unweighted_and_weighted_shares.csv"
    ),
    index=False
)

# Aggregate by country + industry
df_nace_country = df_lfs.groupby(
    ["COUNTRYW", "NACE2_1D_label"]
).agg(agg_dict).reset_index()
df_nace_country.to_csv(
    os.path.join(
        useful_paths.path_eulfs_processed,
        f"eulfs_{year}_by_nace1d_country_final_unweighted_and_weighted_shares.csv"
    ),
    index=False
)

# Aggregate by region + industry
df_nace_region = df_lfs.groupby(
    ["NUTS_ID", "NACE2_1D_label"]
).agg(agg_dict).reset_index()
df_nace_region.to_csv(
    os.path.join(
        useful_paths.path_eulfs_processed,
        f"eulfs_{year}_by_nace1d_region_final_unweighted_and_weighted_shares.csv"
    ),
    index=False
)

#### 4. Inspections for final file

In [22]:
# ------------------------------------------------------------
# 1  Country list for table
# ------------------------------------------------------------
EU_EFTA = {
    "AT","BE","BG","CH","CY","CZ","DE","DK","EE","EL","ES","FI","FR",
    "HR","HU","IE","IS","IT","LT","LU","LV","MT","NL","NO","PL",
    "PT","RO","SE","SI","SK"
}

# ------------------------------------------------------------
# 2  Keep only EU/EFTA rows  +  make key cols *strings*
# ------------------------------------------------------------
df_lfs["COUNTRYW"] = (
    df_lfs["COUNTRYW"].astype(str).str.strip().str[:2]
          .where(lambda s: s.isin(EU_EFTA))
)
df_eu = df_lfs.dropna(subset=["COUNTRYW"]).copy()

cols_to_str = ["COUNTRY", "COUNTRYW", "REGION_2D", "REGION_2DW"]
df_eu[cols_to_str] = df_eu[cols_to_str].apply(lambda s: s.astype(str).str.strip())
df_eu["REGION_2D"]  = df_eu["REGION_2D"].str.zfill(2)
df_eu["REGION_2DW"] = df_eu["REGION_2DW"].str.zfill(2)

# ------------------------------------------------------------
# 3  Helper commuter flags  (as real columns)
# ------------------------------------------------------------
df_eu["CROSS_BORDER"] = df_eu["COUNTRY"]      != df_eu["COUNTRYW"]
df_eu["REG_MOVE"]     = (
        (df_eu["COUNTRY"] == df_eu["COUNTRYW"]) &
        (df_eu["REGION_2D"] != df_eu["REGION_2DW"])
)

# ------------------------------------------------------------
# 4  Build the summary table
# ------------------------------------------------------------
summary_review = (
    df_eu.groupby("COUNTRYW", observed=True)
         .agg(
             n_obs              = ("COUNTRYW", "size"),
             share_total        = ("COUNTRYW", lambda x: 100*len(x)/len(df_eu)),

             # income quality
             pct_with_income    = ("annual_earnings", lambda x: 100*x.notna().mean()),
             income_mean        = ("annual_earnings", "mean"),
             income_median      = ("annual_earnings", "median"),

             # classification coverage
             n_isco             = ("ISCO08_3D",  "nunique"),
             n_nace             = ("NACE2_1D",   "nunique"),
             pct_missing_isco   = ("ISCO08_3D",  lambda x: 100*x.isna().mean()),
             pct_missing_nace   = ("NACE2_1D",   lambda x: 100*x.isna().mean()),
             pct_missing_decile = ("INCDECIL_final", lambda x: 100*x.isna().mean()),
             pct_missing_nuts = ("NUTS_ID", lambda x: 100 * ((x.isna()) | (x.str.len() == 2) | (x.str.endswith("00"))).mean()),

             # commuter counts
             n_cross_border     = ("CROSS_BORDER", "sum"),   # True=1
             n_reg_move         = ("REG_MOVE",     "sum"),
         )
         .assign(
             share_cross_border = lambda d: 100*d.n_cross_border/d.n_obs,
             share_reg_move     = lambda d: 100*d.n_reg_move/d.n_obs,

             # **blank out** mean/median if <2% coverage
             income_mean    = lambda d: d.income_mean.where(d.pct_with_income >= 2),
             income_median  = lambda d: d.income_median.where(d.pct_with_income >= 2),
         )
         .round(2)
         .sort_values("n_obs", ascending=False)
         .reset_index(names="COUNTRYW")
)

# ------------------------------------------------------------
# 5  Friendly column names
# ------------------------------------------------------------
nice_names = {
    "COUNTRYW"          : "Country",
    "n_obs"             : "Observations",
    "share_total"       : "Sample Share (%)",
    "pct_with_income"   : "With Earnings Data (%)",
    "income_mean"       : "Mean Income",
    "income_median"     : "Median Income",
    "n_isco"            : "Occupations (ISCO)",
    "n_nace"            : "Sectors (NACE)",
    "pct_missing_isco"  : "Missing ISCO (%)",
    "pct_missing_nace"  : "Missing NACE (%)",
    "pct_missing_decile": "Missing Decile (%)",
    "pct_missing_nuts"  : "Missing NUTS_ID (%)",   # <- renamed here
    "n_cross_border"    : "Cross-border commuters",
    "share_cross_border": "Cross-border (%)",
    "n_reg_move"        : "Other-region commuters",
    "share_reg_move"    : "Other-region (%)",
}

# ------------------------------------------------------------
# 6  Export
# ------------------------------------------------------------
year    = 2023
out_dir = Path(useful_paths.path_eulfs_interim)

out_dir.mkdir(parents=True, exist_ok=True)   # be safe

summary_review.to_csv(out_dir / f"summary_review_table_{year}.csv", index=False)

latex_code = (summary_review.rename(columns=nice_names)
                             .to_latex(index=False,
                                       float_format="%.2f",
                                       caption=f"Descriptive statistics by country (EU-LFS {year})",
                                       label="tab:descriptive_stats",
                                       longtable=True))
(out_dir / f"summary_review_table_{year}.tex").write_text(latex_code)

summary_review.rename(columns=nice_names).to_excel(
    out_dir / f"summary_review_table_{year}.xlsx", index=False
)

print("Summary files written to", out_dir.resolve())

Summary files written to /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/eurostat_data/interim


Inspect commuters in more detail

In [19]:
# how many REG_MOVE rows where either old or new region is missing?
missing_in_old = df_eu["REG_MOVE"] & df_eu["REGION_2D"].isna()
missing_in_new = df_eu["REG_MOVE"] & df_eu["REGION_2DW"].isna()

print("REG_MOVE with missing REGION_2D:", missing_in_old.sum())
print("REG_MOVE with missing REGION_2DW:", missing_in_new.sum())

# show all distinct (REGION_2D, REGION_2DW) combinations that got flagged
df_eu.loc[df_eu["REG_MOVE"], ["REGION_2D", "REGION_2DW"]] \
     .drop_duplicates() \
     .sort_values(["REGION_2D", "REGION_2DW"])

df_eu[df_eu["REG_MOVE"]].groupby("COUNTRYW")["REG_MOVE"].sum().sort_values(ascending=False)

REG_MOVE with missing REGION_2D: 0
REG_MOVE with missing REGION_2DW: 0


COUNTRYW
AT    85680
DE    80964
HU     9689
IT     6495
CH     4964
SE     3801
BE     3408
DK     3390
PL     2208
RO     2011
SK     1567
FR     1395
HR     1281
ES     1180
CZ     1006
NO      790
FI      550
LT      535
IE      378
BG      166
PT       91
Name: REG_MOVE, dtype: int64

In [29]:
# Count total rows and cross-border commuters
total_rows = len(df_lfs)
cross_border_rows = df_lfs["is_cross_border"].sum()
print(f"Total observations: {total_rows:,}")
print(f"Cross-border commuters: {cross_border_rows:,} ({cross_border_rows / total_rows:.2%})")

Total observations: 1,171,342
Cross-border commuters: 9,822 (0.84%)


In [30]:
# Share of cross-border commuters by country
cross_border_by_country = (
    df_lfs.groupby("COUNTRYW")["is_cross_border"]
    .agg(["count", "sum"])
    .rename(columns={"count": "total", "sum": "cross_border"})
)
cross_border_by_country["share"] = cross_border_by_country["cross_border"] / cross_border_by_country["total"]
cross_border_by_country = cross_border_by_country.sort_values("share", ascending=False)

print("\nCross-border commuter share by country (residence):")
print(cross_border_by_country[["cross_border", "total", "share"]].to_string(float_format="{:.1%}".format))


Cross-border commuter share by country (residence):
          cross_border   total  share
COUNTRYW                             
LU                 887    8650  10.3%
CH                1267   37747   3.4%
AT                1550   87230   1.8%
DE                1328  101250   1.3%
BE                 207   16919   1.2%
FI                 104    9726   1.1%
CZ                 159   16745   0.9%
FR                 145   25587   0.6%
SI                 133   23484   0.6%
LV                  21    4367   0.5%
SK                 156   35605   0.4%
MT                  20    5114   0.4%
NL                 180  102966   0.2%
IS                  12    7827   0.2%
EE                  16   12411   0.1%
HU                 104   87758   0.1%
SE                  72   63427   0.1%
CY                  20   17642   0.1%
ES                  32   33931   0.1%
PT                   8    9095   0.1%
DK                  27   34025   0.1%
IE                   6   10648   0.1%
RO                  33   83834   0.

Inspect regions in more detail

In [17]:
import geopandas as gpd

# ──────────────────────────────────────────────────────────────────────────────
# 1) Load your preprocessed 2023 EU‐LFS
# ──────────────────────────────────────────────────────────────────────────────
year = 2023
fn = f"eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed.csv"
path = os.path.join(useful_paths.path_eulfs_interim, fn)
df = pd.read_csv(path, dtype={"COUNTRYW":"category","REGION_2DW":"category"})
print(f">>> Loaded {len(df)} rows from {fn}\n")

# ──────────────────────────────────────────────────────────────────────────────
# 2) Load the NUTS‐2 geodata (to get the “official” list)
# ──────────────────────────────────────────────────────────────────────────────
# adjust to the shapefile you actually use (3035 vs 4326)
gdf = gpd.read_file(useful_paths.path_geodata_nuts_4326).rename(columns={"NUTS_ID":"REGION_2DW"})
print(f">>> Loaded {len(gdf)} polygons from NUTS shapefile\n")

# ──────────────────────────────────────────────────────────────────────────────
# 3) Per‐country NUTS‐2 code comparison
# ──────────────────────────────────────────────────────────────────────────────
summary = []
# drop any NaNs and cast to str before sorting
countries = df["COUNTRYW"].dropna().astype(str).unique().tolist()
for ctry in sorted(countries):
    codes_data = sorted(
        df.loc[df["COUNTRYW"] == ctry, "REGION_2DW"]
          .dropna()
          .astype(str)
          .unique()
          .tolist()
    )
    codes_geo = sorted(
        gdf.loc[gdf["CNTR_CODE"] == ctry, "REGION_2DW"]
           .astype(str)
           .unique()
           .tolist()
    )
    missing_in_geo  = [c for c in codes_data  if c not in codes_geo]
    missing_in_data = [c for c in codes_geo   if c not in codes_data]
    summary.append({
        "COUNTRY":          ctry,
        "NUTS_in_data":      ", ".join(codes_data),
        "NUTS_in_geodata":   ", ".join(codes_geo),
        "Missing_in_geo":    ", ".join(missing_in_geo),
        "Missing_in_data":   ", ".join(missing_in_data),
        "n_obs_total":       int(len(df[df["COUNTRYW"] == ctry])),
        "n_regions_data":    len(codes_data),
        "n_regions_geodata": len(codes_geo),
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

# ──────────────────────────────────────────────────────────────────────────────
# 4) Optional: count of observations per (country, region)
# ──────────────────────────────────────────────────────────────────────────────
counts = (
    df
    .groupby(["COUNTRYW","REGION_2DW"])
    .size()
    .reset_index(name="n_obs")
    .sort_values(["COUNTRYW","n_obs"], ascending=[True,False])
)
display(counts)


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_96803/4120963657.py:9: DtypeWarning: Columns (4,25,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, dtype={"COUNTRYW":"category","REGION_2DW":"category"})


>>> Loaded 1171342 rows from eu_lfs_merged_2023_with_final_unweighted_shares_and_earnings_incdecil_imputed.csv

>>> Loaded 1798 polygons from NUTS shapefile



,COUNTRY,NUTS_in_data,NUTS_in_geodata,Missing_in_geo,Missing_in_data,n_obs_total,n_regions_data,n_regions_geodata
0,AT,"00, 11, 12, 13, 21, 22, 31, 32, 33, 34","AT, AT1, AT11, AT111, AT112, AT113, AT12, AT12...","00, 11, 12, 13, 21, 22, 31, 32, 33, 34","AT, AT1, AT11, AT111, AT112, AT113, AT12, AT12...",87230,10,48
1,BE,"00, 10, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35","BE, BE1, BE10, BE100, BE2, BE21, BE211, BE212,...","00, 10, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35","BE, BE1, BE10, BE100, BE2, BE21, BE211, BE212,...",16919,12,59
2,BG,"00, 31, 32, 33, 34, 41, 42","BG, BG3, BG31, BG311, BG312, BG313, BG314, BG3...","00, 31, 32, 33, 34, 41, 42","BG, BG3, BG31, BG311, BG312, BG313, BG314, BG3...",13427,7,37
3,CH,"00, 01, 02, 03, 04, 05, 06, 07","CH, CH0, CH01, CH011, CH012, CH013, CH02, CH02...","00, 01, 02, 03, 04, 05, 06, 07","CH, CH0, CH01, CH011, CH012, CH013, CH02, CH02...",37747,8,35
4,CY,00,"CY, CY0, CY00, CY000",00,"CY, CY0, CY00, CY000",17642,1,4
5,CZ,"00, 01, 02, 03, 04, 05, 06, 07, 08","CZ, CZ0, CZ01, CZ010, CZ02, CZ020, CZ03, CZ031...","00, 01, 02, 03, 04, 05, 06, 07, 08","CZ, CZ0, CZ01, CZ010, CZ02, CZ020, CZ03, CZ031...",16745,9,24
6,DE,"00, 11, 12, 13, 14, 21, 22, 23, 24, 25, 26, 27...","DE, DE1, DE11, DE111, DE112, DE113, DE114, DE1...","00, 11, 12, 13, 14, 21, 22, 23, 24, 25, 26, 27...","DE, DE1, DE11, DE111, DE112, DE113, DE114, DE1...",101250,39,455
7,DK,"00, 01, 02, 03, 04, 05","DK, DK0, DK01, DK011, DK012, DK013, DK014, DK0...","00, 01, 02, 03, 04, 05","DK, DK0, DK01, DK011, DK012, DK013, DK014, DK0...",34025,6,18
8,EE,00,"EE, EE0, EE00, EE001, EE004, EE008, EE009, EE00A",00,"EE, EE0, EE00, EE001, EE004, EE008, EE009, EE00A",12411,1,8
9,EL,"00, 30, 41, 42, 43, 51, 52, 53, 54, 61, 62, 63...","EL, EL3, EL30, EL301, EL302, EL303, EL304, EL3...","00, 30, 41, 42, 43, 51, 52, 53, 54, 61, 62, 63...","EL, EL3, EL30, EL301, EL302, EL303, EL304, EL3...",8933,14,70


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_96803/4120963657.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["COUNTRYW","REGION_2DW"])


,COUNTRYW,REGION_2DW,n_obs
6,AT,13,14222
10,AT,31,11077
12,AT,33,10130
8,AT,22,10114
11,AT,32,9655
...,...,...,...
3724,TR,0A,0
3725,TR,81,0
3726,TR,82,0
3727,TR,84,0


In [8]:
out_dir = useful_paths.path_eulfs_interim
out_path = os.path.join(out_dir, "summary_review_2023_nuts2_region_comparison.csv")
summary_df.to_csv(out_path, index=False)

In [39]:
import pandas as pd
import re

# Drop rows with missing NUTS_ID
df_valid_nuts = df_lfs[df_lfs["NUTS_ID"].notna()].copy()

# 1. NUTS_ID basic structure summary
print("\n--- NUTS_ID Structure Summary ---")
df_valid_nuts["NUTS_len"] = df_valid_nuts["NUTS_ID"].apply(len)
nuts_lengths = df_valid_nuts["NUTS_len"].value_counts().sort_index()
print("NUTS_ID lengths:")
print(nuts_lengths)

# Optional: inspect unique patterns (e.g., 2-letter, 4-letter etc.)
print("\nExamples by NUTS_ID length:")
for length in sorted(df_valid_nuts["NUTS_len"].unique()):
    examples = df_valid_nuts[df_valid_nuts["NUTS_len"] == length]["NUTS_ID"].unique()[:5]
    print(f"Length {length}: {examples}")

# 2. Country-wise breakdown
print("\n--- National-level vs Regional NUTS_IDs per Country ---")
summary = (
    df_valid_nuts
    .assign(
        is_national_level=lambda d: d["NUTS_ID"] == d["COUNTRY"]
    )
    .groupby("COUNTRY")
    .agg(
        total_entries=("NUTS_ID", "count"),
        national_code_count=("is_national_level", "sum")
    )
)
summary["regional_code_count"] = summary["total_entries"] - summary["national_code_count"]
summary["national_code_pct"] = (summary["national_code_count"] / summary["total_entries"]) * 100
summary = summary.sort_values("national_code_pct", ascending=False)

print(summary.round(2))

# 3. Show countries with mixed usage (national + regional)
mixed_usage = summary[(summary["national_code_count"] > 0) & (summary["regional_code_count"] > 0)]
if not mixed_usage.empty:
    print("\n⚠️ Countries with mixed NUTS_ID usage (national + regional):")
    print(mixed_usage.round(2))
else:
    print("\n✅ No countries with mixed national and regional NUTS_IDs.")

# 4. Sample problematic entries
print("\n--- Sample entries with national-level NUTS_IDs ---")
print(
    df_valid_nuts[df_valid_nuts["NUTS_ID"] == df_valid_nuts["COUNTRY"]]
    [["COUNTRY", "NUTS_ID", "REGION_2DW", "is_cross_border"]]
    .head(10)
)


--- NUTS_ID Structure Summary ---
NUTS_ID lengths:
NUTS_len
2     108142
4    1060384
Name: count, dtype: int64

Examples by NUTS_ID length:
Length 2: ['MT' 'NL' 'BE' 'DE' 'AT']
Length 4: ['AT13' 'AT11' 'AT12' 'AT31' 'AT22']

--- National-level vs Regional NUTS_IDs per Country ---
         total_entries  national_code_count  regional_code_count  \
COUNTRY                                                            
MT                5094                 5094                    0   
NL              103048               102786                  262   
AT               87090                    0                87090   
BE               17468                    0                17468   
SI               23351                    0                23351   
SE               63357                    0                63357   
RO               83840                    0                83840   
PT                9110                    0                 9110   
PL               79126               

/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_96803/1794682233.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("COUNTRY")


Inspect income

In [37]:
# Columns of interest
income_cols = ["COUNTRYW", "INCDECIL_final", "annual_earnings"]

# Drop rows missing both income fields
df_income = df_lfs[income_cols].dropna(subset=["INCDECIL_final", "annual_earnings"], how='all')

# Distribution of deciles by country
decile_dist = df_income.groupby("COUNTRYW")["INCDECIL_final"].value_counts(normalize=True).unstack(fill_value=0)
decile_dist["n_total"] = df_income.groupby("COUNTRYW").size()

# Earnings summary by country
earnings_stats = df_income.groupby("COUNTRYW")["annual_earnings"].describe().round(1)

# Identify suspicious earnings patterns
suspicious_countries = earnings_stats[
    (earnings_stats["mean"] < 1000) | (earnings_stats["std"] < 1)
]

# Display
print("\n--- Decile Distribution by Country ---")
print(decile_dist)

print("\n--- Annual Earnings Statistics by Country ---")
print(earnings_stats)

print("\n--- Countries with Suspicious Earnings Data ---")
print(suspicious_countries)


--- Decile Distribution by Country ---
INCDECIL_final       1.0       1.5       2.0       2.5       3.0       3.5  \
COUNTRYW                                                                     
AT              0.112493  0.000000  0.079910  0.000000  0.034598  0.019904   
BE              0.027853  0.000000  0.025690  0.123580  0.206328  0.000000   
CH              0.070724  0.000771  0.086975  0.000319  0.094715  0.002181   
CY              0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
CZ              0.055020  0.000000  0.013945  0.000000  0.146805  0.000000   
DE              0.165351  0.000000  0.046955  0.000000  0.345204  0.000000   
DK              0.161254  0.000384  0.131406  0.000177  0.111065  0.001624   
EE              0.092590  0.000324  0.099400  0.002189  0.107183  0.002027   
EL              0.073118  0.000000  0.099862  0.000230  0.117654  0.026860   
ES              0.000000  0.000000  0.000245  0.000000  0.000489  0.114782   
FI              0.070517

#### 5. Saving final file + decisions for simulation

1) General sample

Drop: None

Keep all: AT, BE, BG, CH, CY, CZ, DE, DK, EE, EL, ES, FI, FR, HR, HU, IE, IS, IT, LT, LU, LV, MT, NL, NO, PL, PT, RO, SE, SI, SK

2) Income sample

Drop: ['IT', 'NL', 'DE', 'HU', 'AT', 'RO','PL', 'ES', 'LT', 'SI', 'CY', 'BE','CZ', 'HR', 'BG', 'IS', 'LV', 'MT'] (< 20% earnings data)

Keep: ['SE', 'CH', 'SK', 'DK', 'FR', 'EE', 'EL', IE', 'NO', 'FI', 'PT', 'LU']

In [16]:
import os

year = 2023

# Drop cross-border commuters
df_lfs_clean = df_lfs[df_lfs["is_cross_border"] != True].copy()

# Convert national-level NUTS_IDs for NL and MT to XX00 format
df_lfs_clean.loc[(df_lfs_clean["COUNTRY"] == "NL") & (df_lfs_clean["NUTS_ID"] == "NL"), "NUTS_ID"] = "NL00"
df_lfs_clean.loc[(df_lfs_clean["COUNTRY"] == "MT") & (df_lfs_clean["NUTS_ID"] == "MT"), "NUTS_ID"] = "MT00"

# Save cleaned CSV
output_fname_clean = f"clean_eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed.csv"
final_output_path_clean = os.path.join(useful_paths.path_eulfs_interim, output_fname_clean)
df_lfs_clean.to_csv(final_output_path_clean, index=False)

# Save cleaned Pickle
pkl_fname_clean = f"clean_eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed.pkl"
final_pkl_path_clean = os.path.join(useful_paths.path_eulfs_interim, pkl_fname_clean)
df_lfs_clean.to_pickle(final_pkl_path_clean)

print(f"Saved cleaned files:\n- CSV: {final_output_path_clean}\n- PKL: {final_pkl_path_clean}")

Saved cleaned files:
- CSV: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/eurostat_data/interim/clean_eu_lfs_merged_2023_with_final_unweighted_shares_and_earnings_incdecil_imputed.csv
- PKL: /Users/go82gax/Documents/Projekte/LFS/Analyse/PythonProject/Simulating-Re-Skilling-Journeys/data/eurostat_data/interim/clean_eu_lfs_merged_2023_with_final_unweighted_shares_and_earnings_incdecil_imputed.pkl
